[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/09_causal_attention_solution.ipynb)

# 🔴 Solution: Causal Self-Attention

*Attention & Transformers · Hard*

Reference implementation. Try it yourself in `09_causal_attention.ipynb` first.

---
Implement **causal (autoregressive) self-attention**: position $i$ may attend to
positions $0 \dots i$ and to nothing later.

$$\text{out} = \operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V,
\qquad
M_{ij} = \begin{cases} 0 & j \le i \\ -\infty & j > i \end{cases}$$

### Signature
`causal_attention(Q, K, V)` with `Q`, `K`, `V` all `(..., seq, d)` and the same `seq`
(this is self-attention). Returns `(..., seq, d)`. Leading axes are free, so
`(B, seq, D)` and `(B, H, seq, D_h)` must both work.

### Rules
- No `jax.nn.dot_product_attention`, no `nnx.MultiHeadAttention`, no
  `is_causal=` shortcut from a library
- Build the mask with `jnp.tril` / `jnp.triu` — no Python loop over positions
- The mask is applied to the **scores**, before `jax.nn.softmax`
- Must be jittable and differentiable

### Add before, do not multiply after
The tempting wrong version is:

```python
w = jax.nn.softmax(scores, axis=-1)
w = w * causal          # WRONG
out = w @ V
```

This does not merely zero the future — it corrupts the past. The softmax
denominator was computed over **all** $seq$ positions, including the ones you then
deleted, so row $i$ now sums to $\sum_{j\le i} p_{ij} < 1$ instead of $1$. Two
consequences:

1. **Magnitude collapse.** Row $0$ keeps only $p_{00}$, typically $\approx 1/seq$,
   so the first token's output is shrunk by ~$seq\times$ while the last token's is
   untouched. The layer applies a position-dependent gain that LayerNorm then
   has to undo.
2. **Information leak.** The denominator is a function of the future keys. Even
   with the weights zeroed, $\partial\,\text{out}_0 / \partial k_5 \ne 0$, so a
   language model trained this way is reading its own labels. It will show an
   impossibly low training loss and generate garbage at inference, because at
   decode time the future keys do not exist.

Adding $-\infty$ (or a large negative number) *before* the softmax makes the
blocked logits contribute exactly $0$ to the denominator, so each row is a proper
distribution over its visible prefix and the gradient w.r.t. future keys is
exactly zero.

### Which large negative number
`-1e9` is the usual choice and is fine in `float32` and `bfloat16` — bfloat16
keeps float32's exponent range, so it represents `-1e9` without trouble.
`float16` does not: it tops out at $65504$, so `-1e9` silently becomes `-inf`
there.

That is harmless for a *causal* mask on its own, because every row keeps its own
diagonal and so no row is entirely blocked. It stops being harmless as soon as a
second mask joins in. Combine causal with padding and a sequence that is pure
padding leaves a row with no visible key at all; `softmax` then evaluates
`x - max(x)` as `-inf - (-inf)` = `nan`, which spreads through the whole batch's
gradients. That row is a real-world case, not a contrived one — it is what a
short sequence in a long-padded batch looks like.

Two defensive forms: `jnp.finfo(dtype).min`, which is large but finite in every
dtype, or `jnp.where(mask, scores, -1e9)` — replacing rather than adding, so the
bias can never accumulate. A fully-masked row then produces a uniform
(meaningless, but finite) distribution instead of a `nan`, which is far easier
to debug than a loss that turns to `nan` ten steps later.

### At decode time
With a KV cache, $Q$ has length $1$ while $K$ has length $t+1$ — the mask becomes
all-ones and disappears. If you ever see a `tril` applied to a non-square score
matrix during generation, the alignment is wrong: causality is about absolute
positions, not about the shape of the block you happen to be computing.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def causal_attention(Q, K, V):
    seq, d_k = Q.shape[-2], Q.shape[-1]

    scores = (Q @ jnp.swapaxes(K, -1, -2)) / jnp.sqrt(jnp.asarray(d_k, Q.dtype))

    # allowed[i, j] is True iff j <= i. Shape (seq, seq) broadcasts against any
    # number of leading batch/head axes.
    allowed = jnp.tril(jnp.ones((seq, seq), dtype=bool))

    # Additive bias applied BEFORE the softmax, so blocked logits contribute
    # exactly zero to the denominator.
    bias = jnp.where(allowed, 0.0, -1e9).astype(scores.dtype)
    scores = scores + bias

    weights = jax.nn.softmax(scores, axis=-1)
    return weights @ V

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

seq, D = 5, 4
Q = jax.random.normal(jax.random.key(0), (1, seq, D))
K = jax.random.normal(jax.random.key(1), (1, seq, D))
V = jnp.arange(seq * D, dtype=jnp.float32).reshape(1, seq, D)

# Look at the weight matrix the mask produces.
scores = (Q @ jnp.swapaxes(K, -1, -2)) / jnp.sqrt(float(D))
allowed = jnp.tril(jnp.ones((seq, seq), dtype=bool))
w_right = jax.nn.softmax(scores + jnp.where(allowed, 0.0, -1e9), axis=-1)
w_wrong = jax.nn.softmax(scores, axis=-1) * allowed

print("row sums, mask BEFORE softmax:", w_right.sum(-1)[0])
print("row sums, mask AFTER  softmax:", w_wrong.sum(-1)[0])
print("-> the 'after' version shrinks early positions toward zero")

out = causal_attention(Q, K, V)
print("position 0 output:", out[0, 0], " == V[0]:", V[0, 0])

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("causal_attention")